# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## Task Type: Ranking

My selected lane is Ranking / Recommendation.

The goal is to rank content items based on their expected relevance or usefulness
to users.

Unlike classification, where the output is a category, ranking focuses on ordering
multiple items from most relevant to least relevant.

The system needs to decide which content should appear higher when a user interacts
with the platform.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The ideal target would be the true relevance score of content for each user, but this
information is not directly available.

Therefore, I will use engagement rate as a proxy:

Engagement Rate = clicks_90d / impressions_90d

This proxy represents how effectively content attracts user interaction after being
shown.

A higher engagement rate suggests that content may be more relevant to users.

In [6]:
import pandas as pd

df = pd.read_csv("D:\\flyrank ML\\flyrank-ml-internship-starter\\data\\raw\\content_refresh_anonymized.csv")

df.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


In [7]:
df['engagement_rate'] = (
    df['clicks_90d'] / df['impressions_90d']
)

df[['content_id','client_id','impressions_90d',
    'clicks_90d','engagement_rate']].head()

,content_id,client_id,impressions_90d,clicks_90d,engagement_rate
0,content_304f48230142,client_f369cb89fc,3803,29,0.007626
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.000457
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.000874
3,content_331d6c4de07b,client_19581e27de,11751,58,0.004936
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.001254


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The main success metric will be NDCG@K (Normalized Discounted Cumulative Gain).

NDCG measures how well the ranking system places the most relevant items near the
top of the recommendation list.

A higher NDCG score means the system is better at showing useful content earlier.

This metric is suitable because ranking quality depends on the order of results,
not only whether a prediction is correct.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
ranking_df = df[
[
'content_id',
'client_id',
'impressions_90d',
'clicks_90d',
'pageviews_90d',
'engagement_rate'
]
]

ranking_df.head()

,content_id,client_id,impressions_90d,clicks_90d,pageviews_90d,engagement_rate
0,content_304f48230142,client_f369cb89fc,3803,29,22,0.007626
1,content_a1fb4e703a9e,client_4e07408562,15320,7,10,0.000457
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,14,0.000874
3,content_331d6c4de07b,client_19581e27de,11751,58,87,0.004936
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,177,0.001254


The unit of analysis is:

One row = one content item and its interaction history for a client.

Each row represents how a specific piece of content performed based on user exposure
and engagement.

The model would use these observations to learn which content should receive a
higher ranking.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule might rank content only by total clicks or impressions.

However, simple rules have limitations:
- they ignore relationships between different features,
- they may favor popular content instead of relevant content,
- they cannot adapt to changing user behavior.

Machine learning can combine multiple signals such as:
- engagement,
- competition,
- content characteristics,
- historical performance,

to learn more complex ranking patterns.

Therefore, ML can provide more personalized and flexible ranking decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

✓ I identified my ML task type as Ranking.

✓ I defined a target/proxy variable.

✓ I selected an appropriate success metric.

✓ I explained my unit of analysis.

✓ I connected the model output to a real content ranking action.

✓ I explained why ML is better than a simple rule.